# 01 — Eksploracja danych (Yale EMMLC, etykieta ESI)

Cel: rozkład ESI 1–5 (kolory TOPSOR), braki pomiarów, parametry życiowe i skargi główne.

Wymagania: przygotowane splity (`python scripts/02_preprocess.py` → `data/processed/esi/`
oraz raport `DATA_AUDIT.md`). Temperatura jest już w °C.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.data import ensure_prepared, feature_groups, load_split
from src.data.prepare import META_COLS
from src.data.yale_schema import ARRIVAL_CATEGORIES
from src.utils.config import CLASS_NAMES, CLASS_NAMES_PL, DATA_AUDIT_MD, TOPSOR_HEX

COLORS = [TOPSOR_HEX[c] for c in CLASS_NAMES]   # ESI 1 … ESI 5 (kolory TOPSOR)
ensure_prepared()   # uruchomi przygotowanie danych tylko, gdy splitów brak
groups = feature_groups()
pd.Series({g: len(c) for g, c in groups.items()}, name='liczba cech')

In [ ]:
cols = META_COLS + [c for g in ('demographics', 'arrival', 'time', 'triage_vitals', 'chief_complaints')
                    for c in groups[g]]
df = load_split('train', columns=cols)
print(f'Train: {df.shape}')
df.head()

## Rozkład ESI (kolory TOPSOR)

In [ ]:
counts = df['esi'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(CLASS_NAMES_PL, counts.values, color=COLORS)
ax.set_ylabel('Liczba wizyt')
ax.set_title('Rozkład ESI (train)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
(counts / counts.sum()).round(3)

## Braki pomiarów a poziom ESI

U pacjentów ESI 1 często brakuje wszystkich parametrów: w Yale parametry z triażu to pomiary
sprzed wprowadzenia na salę, więc brak odzwierciedla organizację pracy (pacjent od razu na sali),
czyli decyzję podjętą PO ocenie pilności. Braki zostają NaN, a model dostaje cechy `*_missing`.
W TOPSOR pomiar jest obowiązkowy — dlatego metryki raportujemy też w podgrupie
`vitals_complete == 1` (HR, SBP i RR zmierzone).

In [ ]:
vitals = [c for c in groups['triage_vitals'] if c != 'triage_vital_o2_device']
missing = df[vitals].isna()
by_esi = missing.groupby(df['esi']).mean()
by_esi['wszystkie_brak'] = missing.all(axis=1).groupby(df['esi']).mean()
by_esi.round(3)

## Komplet parametrów (`vitals_complete`) — populacja jak w TOPSOR

Kolumna meta (NIE cecha): 1 = HR, SBP i RR zmierzone w triażu. Ewaluacja w tej podgrupie
przybliża warunki wdrożenia; liczba wizyt ESI 1 z kompletem parametrów jest mała.

In [ ]:
vc = pd.crosstab(df['esi'], df['vitals_complete'], margins=True)
vc['udział_vc'] = (vc[1] / vc['All']).round(3)
display(vc)
df.groupby('site')['vitals_complete'].mean().round(3).rename('udział vc wg oddziału')

## Sposób przybycia wg ESI

Kategorie przenośne na polski SOR: ZRM (`ambulance`), samodzielnie (`self`; także brak danych
w Yale), inny (`other`; m.in. policja).

In [ ]:
arrival = df[[f'arrival_{c}' for c in ARRIVAL_CATEGORIES]]
arrival.groupby(df['esi']).mean().round(3)

## Parametry życiowe wg ESI (temperatura w °C)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, vitals):
    data = [df.loc[df['esi'] == k, col].dropna() for k in range(1, 6)]
    boxes = ax.boxplot(data, showfliers=False, patch_artist=True)
    for patch, color in zip(boxes['boxes'], COLORS):
        patch.set_facecolor(color)
    ax.set_xticks(range(1, 6), [f'ESI {k}' for k in range(1, 6)])
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Najczęstsze skargi główne i rozkład ESI w ich obrębie

In [ ]:
cc = groups['chief_complaints']
top = df[cc].sum().sort_values(ascending=False).head(15).index
table = pd.DataFrame(
    {f'ESI {k}': [df.loc[df[c] > 0, 'esi'].eq(k).mean() for c in top] for k in range(1, 6)},
    index=top,
)
table.insert(0, 'n', df[top].sum().astype(int).values)
table.round(3)

## Hospitalizacja (`admitted`) wg ESI — tylko do analiz, NIE cecha modelu

In [ ]:
df.groupby('esi')['admitted'].mean().round(3)

## Raport z audytu danych

In [ ]:
print(DATA_AUDIT_MD.read_text(encoding='utf-8')[:3000] if DATA_AUDIT_MD.exists()
      else 'Brak DATA_AUDIT.md — uruchom scripts/02_preprocess.py')